# Offline Input audit
Run on the competition L4 machine with Internet disabled and the six private datasets plus Google Gemma 4 model attached. Reads all 168 P33 dependency wheel hashes and the exact native and SDK wheel before any installation. No result is claimed until the cell finishes.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, zipfile
base = Path('/kaggle/input/datasets/waqasm86')
sdk = base / 'kaggle-vllm-sdk-v031-l4-acceptance'
wheel = sdk / 'kaggle_vllm-0.3.1-py3-none-any.whl'
model = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
assert wheel.is_file() and model.joinpath('config.json').is_file()
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''): h.update(block)
    return h.hexdigest()
assert digest(wheel) == 'a14a0be361ea0ab876aa939743ad7259ac012b8c9410cc668899da00b875c27b'
with zipfile.ZipFile(wheel) as z:
    lock = json.loads(z.read('kaggle_vllm/profiles/kaggle-l4x4-cu129/L4_INFERENCE_LOCK.json'))
assert lock['wheel_count'] == len(lock['wheels']) == 168
roots = [base / name for name in ('kaggle-vllm-v030-py312-runtime-core-v2','kaggle-vllm-v030-py312-runtime-large-v2','kaggle-vllm-v030-py312-torch-cu129-v2')]
assert all(root.is_dir() for root in roots)
verified = []
for item in lock['wheels']:
    matches = [p for root in roots for p in root.rglob(item['filename']) if p.is_file()]
    assert len(matches) == 1, (item['filename'], matches)
    path = matches[0]
    assert path.stat().st_size == item['size'] and digest(path) == item['sha256'], path
    verified.append({'filename':item['filename'],'sha256':item['sha256'],'bytes':item['size']})
native = base / 'vllm-0-30-0-cp312-cp312-linux-x86-64' / lock['native']['filename']
assert native.is_file() and native.stat().st_size == lock['native']['size'] and digest(native) == lock['native']['sha256']
input_manifest = json.loads((sdk/'INPUT_MANIFEST.json').read_text())
for name, info in input_manifest['files'].items():
    p = sdk/name
    assert p.is_file() and p.stat().st_size == info['size'] and digest(p) == info['sha256'], name
model_files = sorted(p for p in model.rglob('*') if p.is_file())
assert model_files and any(p.suffix == '.safetensors' for p in model_files)
model_identity = {'path':str(model),'config_sha256':digest(model/'config.json'),'weight_file_count':sum(p.suffix == '.safetensors' for p in model_files),'weight_bytes':sum(p.stat().st_size for p in model_files if p.suffix == '.safetensors')}
gpus = subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],capture_output=True,text=True,check=True,timeout=20).stdout.splitlines()
assert [x.strip() for x in gpus] == ['NVIDIA L4, 8.9'] * 4
report = {'status':'PASS','sdk_sha256':digest(wheel),'native_sha256':digest(native),'dependency_wheels_verified':len(verified),'dependency_total_bytes':sum(x['bytes'] for x in verified),'model':model_identity,'gpus':gpus}
out = Path('/kaggle/working/01_offline_inputs_audit.json')
out.write_text(json.dumps(report,indent=2)+'\n')
print(json.dumps(report,indent=2))
